[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TigerUnderTheMoon/CF/blob/main/examples/01_counterfactual_attribution.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/TigerUnderTheMoon/CF/main/examples/01_counterfactual_attribution.ipynb)

Raw notebook URL: [https://raw.githubusercontent.com/TigerUnderTheMoon/CF/main/examples/01_counterfactual_attribution.ipynb](https://raw.githubusercontent.com/TigerUnderTheMoon/CF/main/examples/01_counterfactual_attribution.ipynb)

# Counterfactual Attribution Quick Example

This notebook loads **10 fixture traces** and demonstrates how `attribution_score` (local counterfactual utility score, 局部反事实效用分数) is computed from reflection steps. It is a diagnostic walkthrough, not a new validation run.

Key terminology (术语解释):
- **attribution_score**: a deterministic mapping from Phase 4 attribution types to scalar scores, reflecting how strongly a reflection step is associated with observed utility.
- **necessity** (Delta-U, 局部效用差异): the outcome difference between the original trace and the ablated trace when one reflection step is removed.
- **structure-preserving ablation** (结构保留消融): an intervention that masks or replaces a reflection span while keeping token count, positional structure, and autoregressive consistency unchanged.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/TigerUnderTheMoon/CF.git"
ROOT = Path.cwd()

# If running in Colab/Kaggle, clone the repository so imports work.
if not (ROOT / "README.md").exists() or not (ROOT / "src" / "fma").exists():
    clone_dir = ROOT / "CF"
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    ROOT = clone_dir

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repository root: {ROOT}")

## 1. Load 10 fixture traces

Fixture traces are stored reasoning trajectories with annotated `reflection_spans` (反思片段，包含 operation_type 和 content).

In [ ]:
import pandas as pd

fixture_path = ROOT / "tests" / "fixtures" / "reflection_traces_subset_10.jsonl"
traces = [json.loads(line) for line in fixture_path.read_text(encoding="utf-8").strip().splitlines()]

print(f"Loaded {len(traces)} fixture traces")
print("Sample keys:", list(traces[0].keys()))
print("First trace reflection_spans:", traces[0]["reflection_spans"])

## 2. Build utility annotations from fixture traces

`UtilityAnnotation` (效用标注) is the Phase 4 dataclass that records whether a reflection step is helpful, neutral, harmful, or spurious. We mock annotations from the fixture data so you can see the full `attribution_score` computation pipeline.

In [ ]:
from fma.eval.utility_annotation import (
    AttributionAlignment,
    OutcomeDelta,
    UtilityAnnotation,
    UtilityLabel,
    utility_annotations_from_records,
)
from fma.eval.counterfactual_attribution import (
    compute_necessity_scores,
    attribution_score_for_annotation,
)

# Map fixture operation_type to attribution_type for demonstration purposes.
TYPE_MAP = {
    "self-reflection": "metacognitive",
    "self-evaluation": "metacognitive",
    "error_diagnosis": "factual_error",
    "plan_revision": "reasoning_gap",
    "strategy_critique": "reasoning_gap",
}

annotations = []
for trace in traces:
    trace_id = trace["trace_id"]
    for idx, span in enumerate(trace.get("reflection_spans", [])):
        op_type = span.get("operation_type", "self-reflection")
        annotations.append(UtilityAnnotation(
            trace_id=trace_id,
            reflection_idx=idx,
            utility=UtilityLabel.HELPFUL,
            outcome_delta=OutcomeDelta.UNCHANGED,
            degradation_score=0.0,
            annotation_confidence=1.0,
            attribution_type=TYPE_MAP.get(op_type, "vague"),
            attribution_alignment=AttributionAlignment.PARTIAL,
            intervention_type="delete",
            reflection_category=trace.get("category", "OTHER"),
            correctness_preserved=trace.get("correctness", True),
        ))

print(f"Created {len(annotations)} utility annotations")

## 3. Compute necessity scores and attribution scores

`compute_necessity_scores` calculates for every reflection step:
- **attribution_score**: the mapped score based on its attribution type.
- **necessity** (Delta-U): `original_utility - ablated_utility`, i.e., how much the trace utility drops when this step is removed.
- **necessity_normalized**: necessity scaled to [0, 1] within each trace.

In [ ]:
scores = compute_necessity_scores(annotations)

rows = []
for s in scores:
    rows.append({
        "trace_id": s.trace_id,
        "step_idx": s.step_idx,
        "attribution_score": s.attribution_score,
        "necessity": s.necessity,
        "necessity_normalized": s.necessity_normalized,
    })

df_scores = pd.DataFrame(rows)
print(f"Computed {len(df_scores)} score rows")
df_scores.head(10)

## 4. Summarize per trace

Aggregate attribution and necessity at the trace level to see which reasoning trajectories carry the highest local utility.

In [ ]:
summary = (
    df_scores.groupby("trace_id", as_index=False)
    .agg(
        mean_attribution_score=("attribution_score", "mean"),
        max_attribution_score=("attribution_score", "max"),
        mean_necessity=("necessity", "mean"),
        max_necessity=("necessity", "max"),
        reflection_steps=("step_idx", "count"),
    )
    .sort_values("max_attribution_score", ascending=False)
)

summary.head(10)

## 5. Visualize local attribution

A horizontal bar chart shows the maximum `attribution_score` per trace among the 10 fixtures.

In [ ]:
import matplotlib.pyplot as plt

plot_data = summary.sort_values("max_attribution_score", ascending=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(plot_data["trace_id"].str.slice(0, 8), plot_data["max_attribution_score"], color="#4C78A8")
ax.set_xlabel("Max attribution_score (最大归因分数)")
ax.set_ylabel("Trace ID")
ax.set_title("Top local attribution among 10 fixture traces\n（10 条 fixture 轨迹中的最高局部归因）")
ax.set_xlim(0, 1)
fig.tight_layout()
plt.show()

## Interpretation

`attribution_score` is **local**: it summarizes a reflection step's observed utility under stored counterfactual ablations (反事实消融). A high value does **not** by itself prove **structural necessity** (结构必要性，即该步骤在图干预后仍被需要) or downstream improvement, or true causal identification (真正的因果识别).

To see how local scores relate to graph structure, continue to [`02_graph_diagnostics.ipynb`](02_graph_diagnostics.ipynb).